# 🤖 Project 3.2 — Robot Joint Logger
**DSA for Mechatronics | Week 03 Project**

---

> **Submission:** Upload your completed `.ipynb` to BlackBoard before the deadline.  
> **Presentation:** You will run and explain your code live in class the following session.  
> **AI tools:** Allowed. You must understand and be able to explain every line you submit.

---

## 🎯 What you are building

A 3-DOF robot arm logs its joint angles (in degrees) every 0.1 seconds during a pick-and-place cycle.
The data is stored as a **2D list**: each row is one time step, each column is one joint.

```
motion_log[t][j]  =  angle of joint j at time step t

Example:
motion_log[0]  = [ 0.00,  0.00,  0.00]   ← t=0: all joints at home
motion_log[1]  = [ 2.31, -1.05,  3.88]   ← t=1: first movements
motion_log[2]  = [ 1.78,  2.14, -0.93]   ← t=2: ...
```

You will build a **motion analysis pipeline** that:
1. Extracts individual joint time-series from the 2D log  
2. Computes cumulative angular displacement using **prefix sums**  
3. Detects travel limit alarms (when a joint has moved too far in total)  
4. Produces a structured log report with formatted timestamps  

---

## 📐 Week 03 concepts you will practise

| Concept | Where used |
|---|---|
| **2D array access** | `log[t][joint_id]` in Exercise 1 |
| **List comprehension** | Extracting columns in Exercise 1 |
| **Prefix sums** | Cumulative displacement in Exercise 2 |
| **O(1) range query** | Travel between two timestamps in Exercise 2 |
| **Linear scan** | First alarm crossing in Exercise 3 |
| **String formatting** | Structured log output in Exercise 4 |

---

## 📊 The prefix sum idea — read this carefully

Without prefix sums, computing the total travel between time `a` and `b` costs **O(b−a)** time.
In a real robot monitoring system querying millions of windows per second, that's too slow.

With prefix sums, any range query costs **O(1)**:

```
displacements = [0, 2.1, 1.8, 2.5, 3.0, 1.2, ...]   ← step-by-step movement
prefix        = [0, 2.1, 3.9, 6.4, 9.4, 10.6, ...]   ← running total

Travel from t=2 to t=4:
  = prefix[4+1] - prefix[2]
  = 9.4 - 3.9
  = 5.5 °   ← computed in O(1), no loop needed!
```

---

## ⚙️ Step 0 — Generate motion data

Run this cell first. Do **not** modify it.


In [ ]:
import random
random.seed(7)

# 150 time steps = 15 seconds at 10 Hz
# Each row: [joint0_angle, joint1_angle, joint2_angle]
# Joints move by small random deltas each step (realistic servo behaviour)

motion_log = []
angles = [0.0, 0.0, 0.0]

for t in range(150):
    deltas = [random.gauss(0, 2.5) for _ in range(3)]
    angles = [round(a + d, 2) for a, d in zip(angles, deltas)]
    motion_log.append(list(angles))

# Preview the data
print(f"motion_log: {len(motion_log)} time steps × {len(motion_log[0])} joints")
print()
print(f"{'t':>4}  {'Joint 0':>9}  {'Joint 1':>9}  {'Joint 2':>9}")
print("─" * 38)
for t in range(8):
    row = motion_log[t]
    print(f"{t:>4}  {row[0]:>9.2f}°  {row[1]:>9.2f}°  {row[2]:>9.2f}°")
print("  ...")

# Show final angles
print()
print("Final angles:")
for j in range(3):
    print(f"  Joint {j}: {motion_log[-1][j]:+.2f}°")


---

## ✏️ Exercise 1 — Extract a joint's time-series

Write `get_joint_series(log, joint_id)` that returns a **flat list** of all angle readings
for one joint across all time steps.

This is purely a 2D array indexing exercise.

### What the function should do:
```
get_joint_series(motion_log, 0)
→ [motion_log[0][0], motion_log[1][0], motion_log[2][0], ...]
→ [0.0, 2.31, 1.78, ...]    ← 150 values
```

> 💡 **One-liner using list comprehension:**
> ```python
> return [log[t][joint_id] for t in range(len(log))]
> ```
> That's really all this function needs to be. Simple but important as a building block.


In [ ]:
def get_joint_series(log, joint_id):
    """
    Extract the angle time-series for one joint from the 2D motion log.
    
    Parameters
    ----------
    log      : list of lists — motion_log[t][j] = angle of joint j at time t
    joint_id : int           — which joint (0, 1, or 2)
    
    Returns
    -------
    list of float — angles for that joint at every time step (length = len(log))
    """
    # YOUR CODE: one list comprehension is enough
    pass


# ── Test ────────────────────────────────────────────────────────────────
j0 = get_joint_series(motion_log, 0)
j1 = get_joint_series(motion_log, 1)
j2 = get_joint_series(motion_log, 2)

print(f"Joint 0 — {len(j0)} readings | first 5: {j0[:5]}")
print(f"Joint 1 — {len(j1)} readings | first 5: {j1[:5]}")
print(f"Joint 2 — {len(j2)} readings | first 5: {j2[:5]}")

# Verify it's equivalent to reading columns directly
assert j0[5] == motion_log[5][0], "Something is wrong — check your indexing"
print("\n✅ Indexing check passed")


---

## ✏️ Exercise 2 — Cumulative displacement (Prefix Sum)

Write `cumulative_displacement(angles)` that returns the **prefix sum of absolute step displacements**.

### Definition:
- Step displacement at time `t` = `abs(angles[t] - angles[t-1])`  (how much the joint moved in one step)
- Cumulative displacement at time `t` = sum of all step displacements from t=0 to t

### The prefix array has length `len(angles) + 1`:
```
prefix[0] = 0                         ← nothing moved yet
prefix[1] = |angles[1] - angles[0]|
prefix[2] = prefix[1] + |angles[2] - angles[1]|
prefix[t] = prefix[t-1] + |angles[t] - angles[t-1]|
```

### O(1) range query:
Total travel between time `a` and `b` (inclusive) = `prefix[b+1] - prefix[a]`

> 💡 **Building prefix sums with a loop:**
> ```python
> prefix = [0]                        # start with 0
> for t in range(1, len(angles)):
>     step = abs(angles[t] - angles[t-1])
>     prefix.append(prefix[-1] + step)
> return prefix
> ```
> `prefix[-1]` is always the last element added — a common Python idiom.


In [ ]:
def cumulative_displacement(angles):
    """
    Returns prefix sum of absolute angular displacements.
    
    Length = len(angles) + 1
    prefix[0]   = 0
    prefix[t+1] = prefix[t] + |angles[t] - angles[t-1]|
    
    Use case: total travel from time a to b = prefix[b+1] - prefix[a]
    """
    prefix = [0]
    
    for t in range(1, len(angles)):
        step = abs(angles[t] - angles[t - 1])
        # YOUR CODE: append running total to prefix
        pass
    
    return prefix


# ── Test ────────────────────────────────────────────────────────────────
j0     = get_joint_series(motion_log, 0)
cum0   = cumulative_displacement(j0)

print(f"Prefix array length : {len(cum0)}  (should be {len(j0) + 1})")
print(f"Total travel Joint 0: {cum0[-1]:.2f}°")
print()

# Demonstrate O(1) range query
for a, b in [(0, 49), (50, 99), (100, 149)]:
    travel = cum0[b + 1] - cum0[a]
    print(f"  Travel t={a:03d} → t={b:03d} : {travel:.2f}°  (O(1) query)")

# Compare all three joints
print()
print("Total travel per joint:")
for j in range(3):
    series = get_joint_series(motion_log, j)
    cum    = cumulative_displacement(series)
    print(f"  Joint {j}: {cum[-1]:.2f}°")


---

## ✏️ Exercise 3 — Travel limit alarm

Write `find_alarm(cum_displacement, limit)` that returns the **first time step** at which
cumulative travel exceeded `limit` degrees, or `None` if it never did.

### Simple linear scan:
```
for t in range(len(cum_displacement)):
    if cum_displacement[t] > limit:
        return t - 1   # ← t-1 because cum[t] corresponds to "after step t-1"
return None
```

> ⚠️ **Index offset:** `cum_displacement` has length `len(angles) + 1`.
> `cum[t]` is the total travel *after* time step `t-1`.
> So if `cum[t] > limit`, the alarm triggers at time step `t - 1`.

> 💡 **Simpler to think about:** loop over `range(1, len(cum))` and return `t - 1` on first hit.

### Expected result with limit=180°:
Joints will alarm at different times depending on how fast they move.


In [ ]:
def find_alarm(cum_displacement, limit=180.0):
    """
    Returns the first time step t where cumulative travel exceeded limit degrees.
    Returns None if the limit is never exceeded.
    
    Parameters
    ----------
    cum_displacement : list of float — prefix sum array (length = n_steps + 1)
    limit            : float         — travel limit in degrees
    
    Returns
    -------
    int or None
    """
    # YOUR CODE: scan through cum_displacement starting at index 1
    # return (t - 1) when cum_displacement[t] > limit
    pass


# ── Test ────────────────────────────────────────────────────────────────
print(f"{'Joint':<8} {'Total travel':>14}  {'Alarm at (180°)':>16}")
print("─" * 44)

for j in range(3):
    series  = get_joint_series(motion_log, j)
    cum     = cumulative_displacement(series)
    total   = cum[-1]
    alarm_t = find_alarm(cum, limit=180.0)
    alarm_str = f"t={alarm_t:03d}" if alarm_t is not None else "never"
    print(f"  Joint {j}   {total:>12.2f}°   {alarm_str:>14}")

# Also try different limits
print()
print("Alarm time for Joint 0 at different limits:")
j0  = get_joint_series(motion_log, 0)
cum = cumulative_displacement(j0)
for lim in [100, 150, 200, 300, 500]:
    t = find_alarm(cum, lim)
    print(f"  limit={lim:>4}° → {'t='+str(t) if t else 'never reached':>12}")


---

## ✏️ Exercise 4 — Full motion log report

Write `generate_log(log, limit, dt)` that calls your three functions and prints
a complete, formatted report.

### Target output format:
```
╔══════════════════════════════════════════════════╗
║          ROBOT ARM MOTION LOG REPORT            ║
╚══════════════════════════════════════════════════╝

  Recording  : 150 steps  (15.0 s @ 10 Hz)
  Joints     : 3

  Angular displacement:
  ┌─────────┬──────────────┬───────────────┐
  │  Joint  │  Total (°)   │  Alarm at     │
  ├─────────┼──────────────┼───────────────┤
  │    0    │   412.33°    │  t=047 (4.7s) │
  │    1    │   388.91°    │  t=051 (5.1s) │
  │    2    │   297.44°    │  never        │
  └─────────┴──────────────┴───────────────┘

  Most active joint  : Joint 0  (412.33°)
  Least active joint : Joint 2  (297.44°)

  ⚠️  2 alarm event(s) detected above 180°
```

> 💡 **Timestamp formatting:**
> ```python
> time_s = t * dt          # e.g. 47 * 0.1 = 4.7 s
> print(f"t={t:03d} ({time_s:.1f}s)")
> ```

> 💡 **Finding most/least active joints:**
> ```python
> totals = [cumulative_displacement(get_joint_series(log, j))[-1] for j in range(3)]
> most_active  = totals.index(max(totals))
> least_active = totals.index(min(totals))
> ```


In [ ]:
def generate_log(log, limit=180.0, dt=0.1):
    """
    Prints a complete motion log report.
    
    Parameters
    ----------
    log   : 2D list — motion_log[t][j]
    limit : float   — travel alarm threshold (degrees)
    dt    : float   — time between steps (seconds)
    """
    n_steps = len(log)
    n_joints = len(log[0])
    duration = n_steps * dt
    
    # --- gather data for all joints ---
    results = []
    for j in range(n_joints):
        series  = get_joint_series(log, j)
        cum     = cumulative_displacement(series)
        total   = cum[-1]
        alarm_t = find_alarm(cum, limit)
        results.append({'joint': j, 'total': total, 'alarm_t': alarm_t})
    
    totals       = [r['total'] for r in results]
    most_active  = totals.index(max(totals))
    least_active = totals.index(min(totals))
    alarm_count  = sum(1 for r in results if r['alarm_t'] is not None)
    
    # YOUR CODE: print the formatted report
    # Use the structure shown above
    # Key pieces:
    #   - box border with ╔ ═ ╗  ║  ╚ ╝
    #   - table rows with │ ─ ├ ┤ ┬ ┴ ┼
    #   - f-strings with alignment e.g. f"{value:>12.2f}"
    #   - for alarm time: f"t={t:03d} ({t*dt:.1f}s)" or "never"
    
    width = 50
    print("╔" + "═"*width + "╗")
    print("║" + "ROBOT ARM MOTION LOG REPORT".center(width) + "║")
    print("╚" + "═"*width + "╝")
    print()
    print(f"  Recording  : {n_steps} steps  ({duration:.1f} s @ {1/dt:.0f} Hz)")
    # ... continue here


# ── Run report ──────────────────────────────────────────────────────────
generate_log(motion_log, limit=180.0, dt=0.1)


---

## 🚀 Stretch Goal — O(1) query demonstration

Add a section to your report that answers 10 random "how far did Joint 0 travel between t=A and t=B?" queries and shows the time taken vs. recomputing from scratch each time.

```python
import time

j0  = get_joint_series(motion_log, 0)
cum = cumulative_displacement(j0)

queries = [(random.randint(0,100), random.randint(101,149)) for _ in range(10)]

# Method 1: O(1) with prefix sum
t0 = time.perf_counter()
results_fast = [cum[b+1] - cum[a] for a, b in queries]
t1 = time.perf_counter()

# Method 2: O(n) recomputation each time
t2 = time.perf_counter()
results_slow = [sum(abs(j0[t]-j0[t-1]) for t in range(a+1,b+1)) for a,b in queries]
t3 = time.perf_counter()

print(f"Prefix sum  : {(t1-t0)*1e6:.2f} µs")
print(f"Recompute   : {(t3-t2)*1e6:.2f} µs")
```
